In [1]:
documents = [
    {
        "id": "GUIDE-001",
        "type": "troubleshooting_guide",
        "content": """
        When a required column is missing from a data pipeline:
        1. Compare the expected and actual column names.
        2. Check whether the upstream source schema changed.
        3. Review recent changes to transformation code.
        4. Verify the approved column mapping.
        """
    },
    {
        "id": "INC-014",
        "type": "historical_incident",
        "content": """
        A previous customer pipeline failed because an upstream
        team changed a column name. The required identifier was
        missing from the incoming data. The team resolved the
        incident by updating the source contract and downstream
        column mapping.
        """
    }
]

In [6]:
from langchain_core.documents import Document

knowledge_documents = [
    Document(
        page_content=doc["content"],
        metadata={
            "id": doc["id"],
            "type": doc["type"]
        }
    )
    for doc in documents
]

In [7]:
knowledge_documents

[Document(metadata={'id': 'GUIDE-001', 'type': 'troubleshooting_guide'}, page_content='\n        When a required column is missing from a data pipeline:\n        1. Compare the expected and actual column names.\n        2. Check whether the upstream source schema changed.\n        3. Review recent changes to transformation code.\n        4. Verify the approved column mapping.\n        '),
 Document(metadata={'id': 'INC-014', 'type': 'historical_incident'}, page_content='\n        A previous customer pipeline failed because an upstream\n        team changed a column name. The required identifier was\n        missing from the incoming data. The team resolved the\n        incident by updating the source contract and downstream\n        column mapping.\n        ')]

In [8]:
print(knowledge_documents[0].page_content)
print("\nMetadata:")
print(knowledge_documents[0].metadata)


        When a required column is missing from a data pipeline:
        1. Compare the expected and actual column names.
        2. Check whether the upstream source schema changed.
        3. Review recent changes to transformation code.
        4. Verify the approved column mapping.
        

Metadata:
{'id': 'GUIDE-001', 'type': 'troubleshooting_guide'}


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\amand\anaconda3\envs\mydatascience\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3218.58it/s]


In [11]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents = knowledge_documents,
    embedding = embeddings
)

In [12]:
query = "Why is the customer ID missing from the incoming data?"

retrieved_docs = vector_store.similarity_search(
    query,
    k=2
)

for doc in retrieved_docs:
    print("Document ID:", doc.metadata["id"])
    print("Document Type:", doc.metadata["type"])
    print("Content:", doc.page_content)
    print("-" * 60)

Document ID: INC-014
Document Type: historical_incident
Content: 
        A previous customer pipeline failed because an upstream
        team changed a column name. The required identifier was
        missing from the incoming data. The team resolved the
        incident by updating the source contract and downstream
        column mapping.
        
------------------------------------------------------------
Document ID: GUIDE-001
Document Type: troubleshooting_guide
Content: 
        When a required column is missing from a data pipeline:
        1. Compare the expected and actual column names.
        2. Check whether the upstream source schema changed.
        3. Review recent changes to transformation code.
        4. Verify the approved column mapping.
        
------------------------------------------------------------


* Right now, FAISS retrieves relevant documents, but our LLM doesn't automatically know about them. Let's connect the retrieved evidence to the diagnosis prompt.

In [13]:
retrieved_context = "\n\n".join(
    [
        f"Source: {doc.metadata['id']} "
        f"({doc.metadata['type']})\n{doc.page_content}"
        for doc in retrieved_docs
    ]
)

print(retrieved_context)

Source: INC-014 (historical_incident)

        A previous customer pipeline failed because an upstream
        team changed a column name. The required identifier was
        missing from the incoming data. The team resolved the
        incident by updating the source contract and downstream
        column mapping.
        

Source: GUIDE-001 (troubleshooting_guide)

        When a required column is missing from a data pipeline:
        1. Compare the expected and actual column names.
        2. Check whether the upstream source schema changed.
        3. Review recent changes to transformation code.
        4. Verify the approved column mapping.
        


In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
from langchain_openai import ChatOpenAI

In [18]:
llm = ChatOpenAI(
    model = "openai/gpt-oss-20b",
    api_key = os.getenv("GROK_API_KEY"),
    base_url = "https://api.groq.com/openai/v1",
)

In [19]:
from pydantic import BaseModel, Field


class PipelineDiagnosis(BaseModel):
    incident_summary: str = Field(
        description="A short summary of the pipeline failure"
    )
    observed_facts: list[str] = Field(
        description="Facts supported by the incident data and diagnostics"
    )
    likely_causes: list[str] = Field(
        description="Possible causes that have not necessarily been confirmed"
    )
    supporting_evidence: list[str] = Field(
        description="Evidence from the incident that supports the analysis"
    )
    recommended_checks: list[str] = Field(
        description="Practical checks an engineer should perform next"
    )
    unknowns: list[str] = Field(
        description="Important information that is still unavailable"
    )

In [20]:
structured_llm = llm.with_structured_output(PipelineDiagnosis)

In [21]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a data pipeline troubleshooting assistant.

        Use the diagnostic findings and retrieved knowledge
        to analyze the incident.

        Rules:
        - Separate verified facts from possible causes.
        - Never present an unverified cause as confirmed.
        - Use retrieved documents as supporting context,
          not as proof that the same cause occurred.
        - Recommend practical checks.
        - Identify important unknowns."""
    ),
    (
        "human",
        """Analyze this pipeline incident.

        Pipeline: {pipeline_name}
        Error: {error}

        Python diagnostic findings:
        {diagnostic_findings}

        Retrieved knowledge:
        {retrieved_context}

        Explain the failure and recommend next steps."""
    )
])

rag_chain = rag_prompt | structured_llm

In [22]:
diagnosis = rag_chain.invoke({
    "pipeline_name": "customer_data_pipeline",
    "error": "KeyError: 'customer_id'",
    "diagnostic_findings": str({
        "status": "failed",
        "missing_columns": ["customer_id"],
        "unexpected_columns": ["id"]
    }),
    "retrieved_context": retrieved_context
})

print(diagnosis.model_dump_json(indent=2))

{
  "incident_summary": "Pipeline customer_data_pipeline failed due to missing required column 'customer_id', causing a KeyError.",
  "observed_facts": [
    "Pipeline status marked as 'failed'.",
    "Diagnostic identified missing column 'customer_id'.",
    "Unexpected column 'id' was present.",
    "Error message: KeyError: 'customer_id'."
  ],
  "likely_causes": [
    "Upstream data source changed column name from 'customer_id' to 'id'.",
    "Column mapping between source and pipeline was not updated.",
    "Transformation code still references 'customer_id' without handling alias."
  ],
  "supporting_evidence": [
    "Python diagnostic findings indicating missing 'customer_id' and presence of 'id'.",
    "Historical incident INC-014 where a similar column rename caused a failure.",
    "Troubleshooting guide recommends checking upstream schema changes."
  ],
  "recommended_checks": [
    "Verify the current upstream source schema to confirm column names.",
    "Compare the expect

In [23]:
new_incident = {
    "pipeline_name": "orders_pipeline",
    "error": "ValueError: could not convert string to float: 'unknown'",
    "diagnostic_findings": {
        "status": "failed",
        "issue": "Invalid numeric value",
        "column": "order_amount",
        "invalid_value": "unknown"
    }
}

new_query = (
    "How should we troubleshoot a pipeline that fails "
    "because order_amount contains the string 'unknown' "
    "instead of a numeric value?"
)

new_docs = vector_store.similarity_search(
    new_query,
    k=2
)

new_context = "\n\n".join(
    f"Source: {doc.metadata['id']} "
    f"({doc.metadata['type']})\n{doc.page_content}"
    for doc in new_docs
)

print("Retrieved sources:")
for doc in new_docs:
    print("-", doc.metadata["id"])

Retrieved sources:
- INC-014
- GUIDE-001


In [24]:
new_diagnosis = rag_chain.invoke({
    "pipeline_name": new_incident["pipeline_name"],
    "error": new_incident["error"],
    "diagnostic_findings": str(
        new_incident["diagnostic_findings"]
    ),
    "retrieved_context": new_context
})

print(new_diagnosis.model_dump_json(indent=2))

{
  "incident_summary": "orders_pipeline failed when attempting to convert the value 'unknown' to a float in the column order_amount, resulting in a ValueError.",
  "observed_facts": [
    "Error: ValueError: could not convert string to float: 'unknown'",
    "Pipeline status 'failed' with issue 'Invalid numeric value'",
    "Diagnostic shows invalid_value 'unknown' in column 'order_amount'"
  ],
  "likely_causes": [
    "The source data contains non-numeric placeholders such as 'unknown' in the order_amount column",
    "An upstream schema change introduced an unexpected string value for order_amount",
    "The transformation code does not handle or sanitize non-numeric values before conversion"
  ],
  "supporting_evidence": [
    "Python diagnostic findings indicating invalid_value 'unknown'",
    "Error message from the pipeline",
    "Historical incident INC-014 describing a similar failure due to upstream column name changes"
  ],
  "recommended_checks": [
    "Verify that the sou

In [25]:
# Same incident for both experiments
comparison_input = {
    "pipeline_name": new_incident["pipeline_name"],
    "error": new_incident["error"],
    "diagnostic_findings": str(
        new_incident["diagnostic_findings"]
    ),
}

# 1. LLM without RAG
baseline_chain = rag_prompt | structured_llm

baseline_diagnosis = baseline_chain.invoke({
    **comparison_input,
    "retrieved_context": (
        "No historical documents were provided. "
        "Use general knowledge and incident evidence only."
    )
})

# 2. LLM with RAG
rag_diagnosis = rag_chain.invoke({
    **comparison_input,
    "retrieved_context": new_context
})

print("========== WITHOUT RAG ==========")
print(baseline_diagnosis.model_dump_json(indent=2))

print("\n========== WITH RAG ==========")
print(rag_diagnosis.model_dump_json(indent=2))

========== WITHOUT RAG ==========
{
  "incident_summary": "orders_pipeline failed due to a non-numeric value 'unknown' in column 'order_amount' causing a ValueError.",
  "observed_facts": [
    "Pipeline status reported as 'failed'.",
    "Error message: ValueError: could not convert string to float: 'unknown'.",
    "Python diagnostic identified 'order_amount' column with invalid_value 'unknown'.",
    "No historical documents or previous incidents referenced."
  ],
  "likely_causes": [
    "Data quality issue where 'unknown' was inserted into a numeric field.",
    "Missing or incorrect data validation before type conversion.",
    "Schema mismatch between source and target expecting numeric values.",
    "Transformation logic attempted to cast string to float without handling non-numeric entries."
  ],
  "supporting_evidence": [
    "Python diagnostic: {'status': 'failed', 'issue': 'Invalid numeric value', 'column': 'order_amount', 'invalid_value': 'unknown'}.",
    "ValueError stac

In [26]:
scored_docs = vector_store.similarity_search_with_score(
    new_query,
    k=2
)

for doc, score in scored_docs:
    print("Document:", doc.metadata["id"])
    print("Distance:", round(score, 4))
    print("Content:", doc.page_content[:150].strip())
    print("-" * 50)

Document: INC-014
Distance: 1.1552
Content: A previous customer pipeline failed because an upstream
        team changed a column name. The required identifier was
        missing from
--------------------------------------------------
Document: GUIDE-001
Distance: 1.2225
Content: When a required column is missing from a data pipeline:
        1. Compare the expected and actual column names.
        2. Check whether the
--------------------------------------------------
